In [ ]:
import torch

# Tensors
Tensors are the building block of PyTorch — every input, weight, activation, and gradient is a tensor.

In [ ]:
# pattern 1.  Python list to Tensor
data = [
    [0, 0, 1, 1, 1, 1, 1, 1, 0, 0],
    [0, 1, 1, 0, 0, 0, 0, 1, 1, 0],
    [1, 1, 0, 1, 0, 0, 1, 0, 1, 1],
    [1, 0, 0, 0, 0, 0, 0, 0, 0, 1],
    [1, 0, 1, 0, 0, 0, 1, 0, 0, 1],
    [1, 0, 0, 1, 1, 1, 1, 0, 0, 1],
    [1, 0, 0, 0, 0, 0, 0, 0, 0, 1],
    [1, 1, 0, 0, 0, 0, 0, 0, 1, 1],
    [0, 1, 1, 1, 1, 1, 1, 1, 1, 0],
    [0, 0, 1, 1, 1, 1, 1, 1, 0, 0]
]
tensors = torch.tensor(data)
print(tensors)

tensor([[0, 0, 1, 1, 1, 1, 1, 1, 0, 0],
        [0, 1, 1, 0, 0, 0, 0, 1, 1, 0],
        [1, 1, 0, 1, 0, 0, 1, 0, 1, 1],
        [1, 0, 0, 0, 0, 0, 0, 0, 0, 1],
        [1, 0, 1, 0, 0, 0, 1, 0, 0, 1],
        [1, 0, 0, 1, 1, 1, 1, 0, 0, 1],
        [1, 0, 0, 0, 0, 0, 0, 0, 0, 1],
        [1, 1, 0, 0, 0, 0, 0, 0, 1, 1],
        [0, 1, 1, 1, 1, 1, 1, 1, 1, 0],
        [0, 0, 1, 1, 1, 1, 1, 1, 0, 0]])


In [ ]:
# pattern 2: creating from a desired shape
# you specify the shape, not the values — this is how you initialize model weights
shape = (2, 3) # a shape tuple for 2 rows and three columns
ones = torch.ones(shape)
zeros = torch.zeros(shape)
random = torch.randn(shape)

print("Ones Tensor", ones)
print("Zeros Tensor", zeros)
print("Random Tensor", random)

Ones Tensor tensor([[1., 1., 1.],
        [1., 1., 1.]])
Zeros Tensor tensor([[0., 0., 0.],
        [0., 0., 0.]])
Random Tensor tensor([[ 1.7012,  0.3837, -0.8498],
        [-0.0247, -0.3012,  3.0407]])


In [ ]:
# pattern 3 creation by mimicking another tensor
# you need a new tensors with exact shape and size
template =  torch.tensor([[1, 2],[3, 4]])
rand_like = torch.randn_like(template, dtype=torch.float)
print("template Tensor", template)
print("rand_like Tensor", rand_like)

template Tensor tensor([[1, 2],
        [3, 4]])
rand_like Tensor tensor([[ 0.0118, -0.4374],
        [ 2.4697,  0.0149]])


# What's inside a tensor
shape, dtype, device

In [ ]:
tensor = torch.randn(2, 3)
print(tensor.shape) # 90% of errors are shape mismatch
print(tensor.dtype) # data type of numbers
print(tensor.device) # where does tensor live

torch.Size([2, 3])
torch.float32
cpu


`torch.float32` is the default dtype for floating-point tensors — a general precision/performance convention across deep learning frameworks, not something specific to gradients. It does matter *for* gradients though: training nudges weights by very small amounts each step, and `float32` has enough precision to represent those small updates without losing them — `float16` can, for example, underflow a small gradient to exactly zero. Autograd is what computes those gradients; the dtype is what determines how precisely they (and the resulting weight updates) can be represented.

# Autograd
- stands for automatic differentiation
- it's torch's built-in gradient calculator
- your model parameters (weights and biases) must be a float type — `float32` is standard.
  Data that represents categories or counts can stay integer.
- it requires being activated by setting `requires_grad = True`

- by default a tensor is just data; to tell PyTorch it's a learnable parameter you must set
  `requires_grad = True` — the single most important setting in all of PyTorch.
- setting it sends a message to the autograd engine: "this is a parameter — from now on,
  track every operation that happens to it."

# data vs parameters


In [ ]:
import torch

# A standard data tensor
x_data = torch.tensor([[1., 2.],
                       [3., 4.]])

# A parameter tensor (we need gradients)
w = torch.tensor([[1.0],
                  [2.0]], requires_grad=True)

print(f"Data tensor requires_grad: {x_data.requires_grad}")
print(f"Parameter tensor requires_grad: {w.requires_grad}")

Data tensor requires_grad: False
Parameter tensor requires_grad: True


Once we set `requires_grad=True`, PyTorch starts a live recording of every operation done on that tensor.

# Building the graph
Goal: compute `z = x * y`, where `y = a + b`

In [ ]:
a = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(3.0, requires_grad=True)
x = torch.tensor(4.0, requires_grad=True)

y = a + b
z = x * y

print(z) # pytorch creates a graph for each operation

tensor(20., grad_fn=<MulBackward0>)


In [ ]:
print(z.grad_fn) # print graph for z
print(y.grad_fn) # print graph for y
print(a.grad_fn) # None — a is a leaf tensor, not the result of an operation

None


In [ ]:
def print_graph(fn, indent=0):
    if fn is None:
        return

    print(" " * indent + str(fn))

    for next_fn, _ in fn.next_functions:
        print_graph(next_fn, indent + 4)


print_graph(z.grad_fn)  # grad_fn is the breadcrumb back through the graph

## Tensor: noun
## Autograd: the nervous system

## Now we need operations — the verbs of torch (the actions/calculations)

## `*` vs `@`

In [ ]:
import torch

# Element-wise multiplication
# Each element is multiplied with the corresponding element.

a = torch.tensor([
    [1, 2],
    [3, 4]
])

b = torch.tensor([
    [10, 20],
    [30, 40]
])

# Element-wise multiplication
result = a * b

print("a:")
print(a)

print("\nb:")
print(b)

print("\na * b:")
print(result)

a:
tensor([[1, 2],
        [3, 4]])

b:
tensor([[10, 20],
        [30, 40]])

a * b:
tensor([[ 10,  40],
        [ 90, 160]])


In [ ]:
# 2. Matrix multiplication — @ powers the neural network linear algebra rule

import torch

m1 = torch.tensor([
    [1, 2, 3],
    [4, 5, 6]
])

m2 = torch.tensor([
    [10, 20],
    [30, 40],
    [50, 60]
])

print("m1 shape:", m1.shape)
print("m2 shape:", m2.shape)

result = m1 @ m2

print("\nm1 @ m2:")
print(result)

print("\nResult shape:", result.shape)

m1 shape: torch.Size([2, 3])
m2 shape: torch.Size([3, 2])

m1 @ m2:
tensor([[220, 280],
        [490, 640]])

Result shape: torch.Size([2, 2])


## Building a linear layer with the classic formula `y = xW + b` uses `@`

## Reduction operations and the `dim` argument

A linear layer's raw output (`y = xW + b`) is one number **per example, per class** — e.g.
a `(2, 3)` tensor for a batch of 2 examples scored against 3 classes. That's not yet a
usable answer: a loss function needs one total number, and a prediction needs one class
per example, not 3 separate scores. **Reduction operations collapse a tensor along one
axis into fewer values** — `sum`/`mean` collapse into an aggregate, `max`/`argmax` collapse
into "the winner, and where it is."

`dim` says *which axis gets collapsed*. For a `(rows, cols)` tensor:
- `dim=0` walks **down** each column, collapsing the row axis — one result per column. **(operates vertically)**
- `dim=1` walks **across** each row, collapsing the column axis — one result per row.**(operates horizonatally)**

So `sum(dim=1)` on a `(2, 3)` tensor removes dim 1 (the column axis) and leaves one value
per row — a `(2,)` result, one total per example.

The four reductions used constantly in classification code:

| Op | Returns | Used for |
|---|---|---|
| `sum` | total of the values along `dim` | e.g. total loss across a batch |
| `mean` | average of the values along `dim` | e.g. average loss per batch |
| `max` | the **largest value** along `dim`, *and* its index — returned together as a `(values, indices)` pair | when you need both the winning score and which class it belongs to |
| `argmax` | **only the index** of the largest value along `dim` — not the value itself | when you only need the predicted class label, which is almost always what you actually want |

`argmax` matters specifically because a classifier's raw output is a score *per class*,
and "the prediction" means "whichever class has the highest score." `argmax(dim=1)` on a
`(batch, num_classes)` tensor returns exactly that: one integer class index per row — e.g.
`[1, 0]` below means "row 0's winning class is index 1, row 1's winning class is index 0."
That index is what gets compared against the true label to check if the model was right.

In [ ]:
scores = torch.tensor([
    [2.0, 5.0, 1.0],
    [7.0, 0.5, 3.0],
])

print("scores:\n", scores)
print("\nsum over everything:", scores.sum())
print("sum dim=0 (collapse rows, down each column):", scores.sum(dim=0))
print("sum dim=1 (collapse columns, across each row):", scores.sum(dim=1))
print("\nmean dim=1:", scores.mean(dim=1))

scores:
 tensor([[2.0000, 5.0000, 1.0000],
        [7.0000, 0.5000, 3.0000]])

sum over everything: tensor(18.5000)
sum dim=0 (collapse rows, down each column): tensor([9.0000, 5.5000, 4.0000])
sum dim=1 (collapse columns, across each row): tensor([ 8.0000, 10.5000])

mean dim=1: tensor([2.6667, 3.5000])


In [ ]:
max_vals, max_idx = scores.max(dim=1)
print("\nmax dim=1 -> values:", max_vals, " indices:", max_idx)
print("argmax dim=1 (predicted class per row, in classification):", scores.argmax(dim=1))

# keepdim=True keeps the collapsed axis around as size 1 instead of dropping it entirely —
# matters when the result needs to still broadcast against the original (batch, classes)
# tensor, e.g. dividing each row by its own sum.
print("\nshape with keepdim=False:", scores.sum(dim=1, keepdim=False).shape)
print("shape with keepdim=True: ", scores.sum(dim=1, keepdim=True).shape)


max dim=1 -> values: tensor([5., 7.])  indices: tensor([1, 0])
argmax dim=1 (predicted class per row, in classification): tensor([1, 0])

shape with keepdim=False: torch.Size([2])
shape with keepdim=True:  torch.Size([2, 1])


In [ ]:
import torch

# Create a 1-D tensor
x = torch.tensor([10, 30, 20, 40, 15])

# argmax() returns the INDEX/POSITION of the largest value
# Values:  10   30   20   40   15
# Index:    0    1    2    3    4
#                       ↑
#                    largest
print(torch.argmax(x))   # tensor(3)

# If you want the actual largest VALUE, use max()
print(torch.max(x))      # tensor(40)

tensor(3)
tensor(40)


In [ ]:
import torch

# ---------------------------------------------------------
# Softmax
# ---------------------------------------------------------
# Definition:
# softmax converts raw scores (logits) into probabilities.
# All output probabilities are between 0 and 1
# and their sum is equal to 1.
#
# Formula:
# softmax(x_i) = exp(x_i) / sum(exp(x_j))
# ---------------------------------------------------------

# Raw model scores (logits)
x = torch.tensor([2.0, 1.0, 0.1])

# Apply softmax along dimension 0
probabilities = torch.softmax(x, dim=0)

print("Logits:        ", x)
print("Probabilities: ", probabilities)
print("Sum:           ", probabilities.sum())

# Find the class with the highest probability
prediction = torch.argmax(probabilities)

print("Predicted class:", prediction)

# Expected:
# Logits:         tensor([2.0000, 1.0000, 0.1000])
# Probabilities:  tensor([0.6590, 0.2424, 0.0986])
# Sum:            tensor(1.)
# Predicted class: tensor(0)

Logits:         tensor([2.0000, 1.0000, 0.1000])
Probabilities:  tensor([0.6590, 0.2424, 0.0986])
Sum:            tensor(1.0000)
Predicted class: tensor(0)


## Softmax: turning raw scores into probabilities

`argmax` above gives a **hard decision** — just the index of the winning class, with no
sense of *how confident* that decision was. A row of raw scores like `[2.0, 5.0, 1.0]` and
one like `[2.0, 5.0, 4.9]` both `argmax` to class 1, but the first is a clear win and the
second is nearly a tie between classes 1 and 2 — `argmax` alone can't tell those two cases
apart. Raw scores themselves aren't usable as confidence either: they aren't bounded to
`[0, 1]` and don't sum to anything meaningful (`2.0 + 5.0 + 1.0 = 8.0` here, but a
different row could sum to anything), so they can't be compared or interpreted as
probabilities as-is.

**Softmax fixes this** by turning a row of raw scores into a genuine probability
distribution, in two steps:
1. **Exponentiate** every score (`e^score`) — this makes every value positive, and because
   `e^x` grows faster than `x`, it *disproportionately* boosts the largest score relative
   to the others (a small lead in raw score becomes a much bigger lead after exponentiating
   — this is the "with the largest score dominating the distribution" part).
2. **Normalize** by dividing each exponentiated score by the row's total — this is exactly
   the `sum(dim=1, keepdim=True)` reduction from above, forcing each row to sum to 1.

The output is now a real probability per class. Critically, `argmax` on these
probabilities always agrees with `argmax` on the raw scores — exponentiating (always
increasing) and dividing by a positive constant (the row sum) never changes *which* value
is largest, only the scale. So softmax doesn't change the prediction `argmax` already
gave you; it adds the magnitude/confidence information `argmax` was missing — the
"5.0 vs 4.9" near-tie above becomes visibly close in probability space (e.g. ~55% vs ~45%),
while a genuinely confident row stays confident (e.g. ~95% vs ~5%).

In [ ]:
import torch.nn.functional as F

probs = F.softmax(scores, dim=1)   # dim=1 -> normalize across each row's classes
print("probs:\n", probs)
print("\neach row sums to 1:", probs.sum(dim=1))
print("argmax on probs matches argmax on raw scores:", probs.argmax(dim=1), scores.argmax(dim=1))

probs:
 tensor([[0.0466, 0.9362, 0.0171],
        [0.9806, 0.0015, 0.0180]])

each row sums to 1: tensor([1.0000, 1.0000])
argmax on probs matches argmax on raw scores: tensor([1, 0]) tensor([1, 0])


## Arange and Reshape

In [ ]:
x = torch.arange(12)
print(x)

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11])


In [ ]:
x = torch.arange(12).reshape(3, 4)
print(x)

tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])


In [ ]:
# getting thirs columns
col_2 = x[:, 2]
print(col_2)

tensor([ 2,  6, 10])


In [ ]:
import torch

# Each row represents one example.
# Each column represents a class.
scores = torch.tensor([
    [10, 0, 5, 20, 1],   # Example 1 → highest score is 20 at index 3
    [1, 30, 2, 5, 0]     # Example 2 → highest score is 30 at index 1
])

# dim=1 means: look across each ROW
# and find the INDEX of the highest value.
#
# Row 0: [10, 0, 5, 20, 1] → max = 20 → index = 3
# Row 1: [1, 30, 2, 5, 0]  → max = 30 → index = 1
best_indices = torch.argmax(scores, dim=1)

print(best_indices)

# Output:
# tensor([3, 1])
#
# Meaning:
# Example 1 → predicted class 3
# Example 2 → predicted class 1

tensor([3, 1])


# torch.gather

cornerstone for complex architecture

In [ ]:
import torch

# Create a 2D tensor.
# Each row represents one example.
# Each column represents a value we may want to select.
data = torch.tensor([
    [10, 11, 12, 13],   # row 0
    [20, 21, 22, 23],   # row 1
    [30, 31, 32, 33]    # row 2
])

# Tell torch.gather which column to select from EACH row.
#
# Row 0 → index 2 → data[0][2] = 12
# Row 1 → index 0 → data[1][0] = 20
# Row 2 → index 3 → data[2][3] = 33
#
# Shape must match the dimension we are gathering from.
indices_to_select = torch.tensor([
    [2],   # select column 2 from row 0
    [0],   # select column 0 from row 1
    [3]    # select column 3 from row 2
])

# dim=1 means we are gathering ALONG the columns.
#
# In simple terms:
# "For each row, go to the column specified by indices_to_select."
selected_values = torch.gather(
    data,
    dim=1,
    index=indices_to_select
)

print(selected_values)

# Output:
# tensor([
#     [12],   # row 0, column 2
#     [20],   # row 1, column 0
#     [33]    # row 2, column 3
# ])

tensor([[12],
        [20],
        [33]])
